In [1]:
# =========================================
# 03 — RFM Segmentation (Olist) | Colab
# Reads orders_fact_light, computes RFM, assigns segments, exports to Drive
# =========================================

import os
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# --- Paths ---
EXPORT_DIR = "/content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports"
ORDERS_LIGHT_PATH = os.path.join(EXPORT_DIR, "orders_fact_light.csv")

if not os.path.exists(ORDERS_LIGHT_PATH):
    raise FileNotFoundError(f"Could not find: {ORDERS_LIGHT_PATH}")

In [3]:
# --- Load data ---
orders = pd.read_csv(ORDERS_LIGHT_PATH)

In [4]:
# --- Parse dates ---
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)

In [5]:
# Keep only valid rows
orders = orders.dropna(subset=["customer_unique_id", "order_purchase_timestamp"]).copy()

print("orders shape:", orders.shape)
print("date range:", orders["order_purchase_timestamp"].min(), "to", orders["order_purchase_timestamp"].max())

orders shape: (96478, 6)
date range: 2016-09-15 12:16:38 to 2018-08-29 15:00:37


In [6]:
# -------------------------------------------------
# 1) Define analysis date
# -------------------------------------------------
# Standard approach: use max purchase date + 1 day
analysis_date = orders["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

print("analysis_date:", analysis_date)

analysis_date: 2018-08-30 15:00:37


In [7]:
# -------------------------------------------------
# 2) Build RFM table
# -------------------------------------------------
rfm = (
    orders.groupby("customer_unique_id", as_index=False)
    .agg(
        last_purchase_date=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("revenue", "sum")
    )
)

rfm["recency_days"] = (analysis_date - rfm["last_purchase_date"]).dt.days

# Optional: reorder columns
rfm = rfm[
    ["customer_unique_id", "recency_days", "frequency", "monetary", "last_purchase_date"]
].copy()

print("\nRFM preview:")
print(rfm.head())


RFM preview:
                 customer_unique_id  recency_days  frequency  monetary  \
0  0000366f3b9a7992bf8c76cfdf3221e2           112          1    141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f           115          1     27.19   
2  0000f46a3911fa3c0805444483337064           537          1     86.22   
3  0000f6ccb0745a6a4b88665a16c9f078           321          1     43.62   
4  0004aac84e0df4da2b147fca70cf8255           288          1    196.89   

   last_purchase_date  
0 2018-05-10 10:56:27  
1 2018-05-07 11:11:27  
2 2017-03-10 21:05:03  
3 2017-10-12 20:29:41  
4 2017-11-14 19:45:42  


In [8]:
# -------------------------------------------------
# 3) Create R, F, M scores (1 to 5)
# -------------------------------------------------
# Recency: lower is better
# Frequency: higher is better
# Monetary: higher is better

# Handle duplicate values in quantiles with rank fallback when needed
rfm["recency_rank"] = rfm["recency_days"].rank(method="first")
rfm["frequency_rank"] = rfm["frequency"].rank(method="first")
rfm["monetary_rank"] = rfm["monetary"].rank(method="first")

rfm["R_score"] = pd.qcut(rfm["recency_rank"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency_rank"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary_rank"], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Combined code
rfm["RFM_score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

# Simple numeric total (optional, useful in BI)
rfm["RFM_total"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

In [9]:
# -------------------------------------------------
# 4) Assign CRM-friendly segments
# -------------------------------------------------
def assign_rfm_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 4:
        return "Loyal Customers"
    elif r >= 4 and f >= 2 and m >= 2:
        return "Potential Loyalists"
    elif r == 5 and f == 1:
        return "New Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Hibernating"
    else:
        return "Needs Attention"

rfm["rfm_segment"] = rfm.apply(assign_rfm_segment, axis=1)

In [10]:
# -------------------------------------------------
# 5) Segment summary (optional for quick inspection)
# -------------------------------------------------
segment_summary = (
    rfm.groupby("rfm_segment", as_index=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        avg_recency_days=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
        total_revenue=("monetary", "sum")
    )
    .sort_values(by="customers", ascending=False)
)

print("\nSegment summary:")
print(segment_summary)


Segment summary:
           rfm_segment  customers  avg_recency_days  avg_frequency  \
0              At Risk      22357        393.942703       1.048173   
4      Needs Attention      17934        180.180718       1.000000   
3      Loyal Customers      15969        150.721085       1.054794   
2          Hibernating      14986        395.508141       1.000000   
6  Potential Loyalists      11937         90.945547       1.000000   
1            Champions       6455         90.554764       1.180945   
5        New Customers       3720         45.949462       1.000000   

   avg_monetary  total_revenue  
0    166.903611     3731464.03  
4    135.893023     2437105.48  
3    115.584070     1845762.01  
2    162.936046     2441759.58  
6    196.936042     2350825.53  
1    312.133709     2014823.09  
5    160.761836      598034.03  


In [11]:
# -------------------------------------------------
# 6) Export
# -------------------------------------------------
rfm_path = os.path.join(EXPORT_DIR, "rfm_customers.csv")
segment_summary_path = os.path.join(EXPORT_DIR, "rfm_segment_summary.csv")

rfm.to_csv(rfm_path, index=False)
segment_summary.to_csv(segment_summary_path, index=False)

print("\n✅ Exported:", rfm_path)
print("✅ Exported:", segment_summary_path)

rfm.head()


✅ Exported: /content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports/rfm_customers.csv
✅ Exported: /content/drive/MyDrive/Analytics_Portfolio/Olist Case/exports/rfm_segment_summary.csv


,customer_unique_id,recency_days,frequency,monetary,last_purchase_date,recency_rank,frequency_rank,monetary_rank,R_score,F_score,M_score,RFM_score,RFM_total,rfm_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,2018-05-10 10:56:27,22405.0,1.0,59249.0,4,1,4,414,9,Needs Attention
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,2018-05-07 11:11:27,23397.0,2.0,2022.0,4,1,1,411,6,Needs Attention
2,0000f46a3911fa3c0805444483337064,537,1,86.22,2017-03-10 21:05:03,90061.0,3.0,36834.0,1,1,2,112,4,Hibernating
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2017-10-12 20:29:41,66546.0,4.0,11437.0,2,1,1,211,4,Hibernating
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2017-11-14 19:45:42,61854.0,5.0,72819.0,2,1,4,214,7,Hibernating
